In [22]:
!git clone https://github.com/imets01/synthetic_network_data_gen.git

Cloning into 'synthetic_network_data_gen'...
remote: Enumerating objects: 776, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 776 (delta 34), reused 35 (delta 24), pack-reused 721 (from 4)
Receiving objects: 100% (776/776), 557.84 MiB | 42.58 MiB/s, done.
Resolving deltas: 100% (407/407), done.
Updating files: 100% (39/39), done.


In [23]:
%cd synthetic_network_data_gen
!git fetch
!git checkout anna

/home/ubuntu/sequence generation/synthetic_network_data_gen/synthetic_network_data_gen/synthetic_network_data_gen
branch 'anna' set up to track 'origin/anna'.
Switched to a new branch 'anna'


In [1]:
high_level_csv = '/home/ubuntu/sequence generation/synthetic_network_data_gen/high_level_features/dataset/all_captures_dataset.csv'
low_level_dir = '/home/ubuntu/sequence generation/synthetic_network_data_gen/low_level_features/dataset/separate_low_level_files.zip'

checkpoint_path = '/home/ubuntu/sequence generation/checkpoints/'

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import zipfile
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import torch.autograd as autograd

# --- WGAN-GP Specific Hyperparameters ---
CRITIC_ITERATIONS = 5  # Train the critic 5 times for every 1 generator training
LAMBDA_GP = 10         # The gradient penalty lambda, a standard value

class QuicSequenceDataset(Dataset):
    def __init__(self, high_level_csv, low_level_zip_path, folder_name_in_zip='separate_low_level_files'):

        self.high_level_df = pd.read_csv(high_level_csv)
        self.high_level_df = self.high_level_df.replace([np.inf, -np.inf], np.nan).dropna(how='any')
        
        # Use a copy of flow_ids as a string Series for reliable mapping/filtering
        self.flow_ids_int = self.high_level_df['file_id'].copy()
        
        self.low_level_zip_path = low_level_zip_path
        self.folder_name_in_zip = folder_name_in_zip

        to_keep = [
            'implementation', 'connection_duration',  'version_negotiation_occurred', 'retry_occurred', 'migration_type',
            'first_path_validation_response_latency', 'path_validation_initiated', 'packets_sent_client', 
            'packets_sent_server', 'handshake_duration', 'time_to_migration', 'migration_duration', # Corrected
            'packets_before_migration', 'total_bidi_streams_client_init',
            'total_udi_streams_client_init',
            # 'connection_close_type'
        ]
        self.condition_features_df = self.high_level_df.copy() # Work on a copy

        for col in to_keep:
            if col not in self.condition_features_df.columns:
                 self.condition_features_df[col] = 0 

        self.condition_features_df = self.condition_features_df[to_keep + ['file_id']]

        categorical_cols = ['migration_type', 'implementation']
        existing_categorical_cols = [col for col in categorical_cols if col in self.condition_features_df.columns]
        for col in existing_categorical_cols:
            if self.condition_features_df[col].dtype == 'object':
                self.condition_features_df[col] = self.condition_features_df[col].astype('category')

        categorical_cols_to_dummy = self.condition_features_df.select_dtypes(include=['category']).columns
        self.condition_features_df = pd.get_dummies(self.condition_features_df, columns=categorical_cols_to_dummy, prefix=categorical_cols_to_dummy)

        
        self.condition_scaler = MinMaxScaler(feature_range=(-1, 1))
        self.sequence_scaler = MinMaxScaler(feature_range=(-1, 1))

        # New list to hold ALL sequence TENSORS in memory
        self.preloaded_sequences = [] 
        self.flow_id_to_path = {}
        self.sequence_columns = None
        
        # The list of flow IDs we need to process
        target_flow_ids = set(self.flow_ids_int.astype(str))

        # 1. Create temporary lists to hold validated data
        temp_sequences = []
        temp_valid_ids = []
        dropped_count = 0

        with zipfile.ZipFile(self.low_level_zip_path, 'r') as zf:
            zip_names = zf.namelist()
            target_prefix = f"{self.folder_name_in_zip}/"
            
            # Map potential files first
            potential_files = {}
            for name in zip_names:
                if name.startswith(target_prefix) and name.endswith('.csv'):
                    relative_name = name[len(target_prefix):] 
                    flow_id_match = relative_name.split('_')[0]
                    if flow_id_match in target_flow_ids:
                        potential_files[int(flow_id_match)] = name
                        target_flow_ids.discard(flow_id_match)

            potential_ids = list(potential_files.keys())

            # Loop through potential IDs, load data, CHECK VALIDITY, then store
            for flow_id in tqdm(potential_ids, desc="Loading & Validating Sequences"):
                file_name_in_zip = potential_files[flow_id]
                
                with zf.open(file_name_in_zip) as f:
                    try:
                        df = pd.read_csv(f).drop('frame_number', axis=1)
                        vals = df.values

                        # --- NEGATIVE VALUE CHECK ---
                        if (vals < 0).any():
                            dropped_count += 1
                            continue # Skip this sequence and do not add to valid lists
                        # ----------------------------

                        if self.sequence_columns is None: 
                            self.sequence_columns = df.columns.tolist()
                        
                        # Only add if check passed
                        temp_sequences.append(vals)
                        temp_valid_ids.append(flow_id)
                        self.flow_id_to_path[flow_id] = file_name_in_zip

                    except Exception as e:
                        print(f"Error processing {file_name_in_zip}: {e}")
                        continue

        print(f"Dropped {dropped_count} sequences containing negative values.")

        # 2. Update the High-Level Dataframe to match valid sequences
        # This removes the rows (conditions) corresponding to the dropped negative sequences
        self.flow_ids = pd.Series(temp_valid_ids)
        self.condition_features_df = self.condition_features_df[self.condition_features_df['file_id'].isin(temp_valid_ids)].reset_index(drop=True)
        
        # 3. Fit scaler on ALL data
        if temp_sequences:
            full_sequence_data = np.concatenate(temp_sequences, axis=0)
            
            if np.isnan(full_sequence_data).any() or np.isinf(full_sequence_data).any():
                print("FATAL ERROR: NaN or INF found in the unscaled sequence data.")
            
            print(f"Unscaled Sequence Data Min/Max: {full_sequence_data.min():.4f} / {full_sequence_data.max():.4f}")
            self.sequence_scaler.fit(full_sequence_data)

            # Fit condition scaler on the filtered condition dataframe
            condition_features_for_scaling = self.condition_features_df.drop('file_id', axis=1)
            self.scaled_conditions = self.condition_scaler.fit_transform(condition_features_for_scaling)
            self.scaled_conditions = torch.FloatTensor(self.scaled_conditions)
            
            # Transform sequences to tensors
            for unscaled_seq in tqdm(temp_sequences, desc="Scaling Sequences to Tensors"):
                scaled_seq = self.sequence_scaler.transform(unscaled_seq)
                if scaled_seq.min() < -1.01 or scaled_seq.max() > 1.01:
                    print("WARNING: SCALED DATA OUT OF RANGE!")
                self.preloaded_sequences.append(torch.FloatTensor(scaled_seq))
        else:
            print("Error: No valid sequence data found!")
            return
        
        # if self.condition_features_df.isnull().values.any() or np.isinf(self.condition_features_df.values).any():
        #     print("FATAL ERROR: NaN or INF found in the condition data.")

    def __len__(self):
        return len(self.preloaded_sequences)

    def __getitem__(self, idx):
        scaled_sequence = self.preloaded_sequences[idx]
        scaled_condition = self.scaled_conditions[idx]
        
        return {
            'sequence': scaled_sequence, 
            'condition': scaled_condition 
        }

In [3]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    device = "cuda"
else:
    print("GPU not available, falling back to CPU.")
    device = "cpu"

GPU is available!


In [4]:
def collate_fn(batch):
    sequences = [item['sequence'] for item in batch]
    conditions = torch.stack([item['condition'] for item in batch])
    padded_sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0.0)
    return {'sequence': padded_sequences, 'condition': conditions}

In [5]:
class Generator(nn.Module):
    def __init__(self, latent_dim, condition_dim, sequence_feature_dim, hidden_dim=64, num_layers=1):
        super().__init__()
        self.rnn = nn.LSTM(input_size=condition_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_init_h = nn.Linear(latent_dim + condition_dim, hidden_dim * num_layers)
        self.fc_init_c = nn.Linear(latent_dim + condition_dim, hidden_dim * num_layers)
        self.fc_out = nn.Linear(hidden_dim, sequence_feature_dim)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, noise, condition, seq_len):
        batch_size = noise.shape[0]
        combined_input = torch.cat([noise, condition], dim=1)
        h0 = self.fc_init_h(combined_input).view(self.num_layers, batch_size, self.hidden_dim)
        c0 = self.fc_init_c(combined_input).view(self.num_layers, batch_size, self.hidden_dim)
        initial_state = (h0, c0)
        condition_expanded = condition.unsqueeze(1).repeat(1, seq_len, 1)
        rnn_out, _ = self.rnn(condition_expanded, initial_state)

        output = torch.tanh(self.fc_out(rnn_out))
        return output

class Critic(nn.Module):
    def __init__(self, condition_dim, sequence_feature_dim, hidden_dim=64, num_layers=1):
        super().__init__()
        self.rnn = nn.LSTM(input_size=sequence_feature_dim + condition_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, 1) # Outputs a single score

    def forward(self, sequence, condition):
        seq_len = sequence.shape[1]
        condition_expanded = condition.unsqueeze(1).repeat(1, seq_len, 1)
        combined_input = torch.cat([sequence, condition_expanded], dim=2)
        _, (hn, cn) = self.rnn(combined_input)
        out = self.fc_out(hn[-1])

        return out

In [6]:
def compute_gradient_penalty(critic, real_samples, fake_samples, condition, device):
    alpha = torch.randn(real_samples.size(0), 1, 1, device=device)
    alpha = alpha.expand_as(real_samples)

    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)

    # --- Temporarily disable cuDNN for the double backward pass ---
    with torch.backends.cudnn.flags(enabled=False):
        d_interpolates = critic(interpolates, condition)

    fake = torch.ones(d_interpolates.size(), device=device)

    gradients = autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.reshape(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [7]:
import os
import torch

def save_checkpoint(generator, critic, g_optimizer, d_optimizer, epoch, g_loss, d_loss, path="checkpoint.pth"):
    checkpoint = {
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'critic_state_dict': critic.state_dict(),
        'g_optimizer_state_dict': g_optimizer.state_dict(),
        'd_optimizer_state_dict': d_optimizer.state_dict(),
        'g_loss': g_loss,
        'd_loss': d_loss,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved at epoch {epoch+1}")

def load_checkpoint(generator, critic, g_optimizer, d_optimizer, path="checkpoint.pth", device='cpu'):
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location=device)
        generator.load_state_dict(checkpoint['generator_state_dict'])
        critic.load_state_dict(checkpoint['critic_state_dict'])
        g_optimizer.load_state_dict(checkpoint['g_optimizer_state_dict'])
        d_optimizer.load_state_dict(checkpoint['d_optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resumed from epoch {start_epoch}")
        return start_epoch
    else:
        print("No checkpoint found, starting from scratch")
        return 0

def train_wgan_gp(dataloader, generator, critic, g_optimizer, d_optimizer, device, epochs=100, latent_dim=10):

    start_epoch = load_checkpoint(generator, critic, g_optimizer, d_optimizer,path=checkpoint_path, device=device)

    for epoch in range(start_epoch, epochs):
        for i, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")):
            real_sequences = batch['sequence'].to(device)
            conditions = batch['condition'].to(device)
            batch_size, seq_len, _ = real_sequences.shape

            for _ in range(CRITIC_ITERATIONS):
                d_optimizer.zero_grad()
                noise = torch.randn(batch_size, latent_dim, device=device)
                fake_sequences = generator(noise, conditions, seq_len)

                real_validity = critic(real_sequences, conditions)
                fake_validity = critic(fake_sequences.detach(), conditions)

                gradient_penalty = compute_gradient_penalty(
                    critic, real_sequences.data, fake_sequences.data, conditions.data, device
                )
                d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA_GP * gradient_penalty
                d_loss.backward()
                d_optimizer.step()

            g_optimizer.zero_grad()
            gen_sequences = generator(noise, conditions, seq_len)
            g_loss = -torch.mean(critic(gen_sequences, conditions))
            g_loss.backward()
            g_optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}], Critic Loss: {d_loss.item():.4f}, Generator Loss: {g_loss.item():.4f}")

        if (epoch + 1) % 200 == 0:
            save_checkpoint(generator, critic, g_optimizer, d_optimizer, epoch, g_loss.item(), d_loss.item(), path=checkpoint_path)


In [8]:
def print_gpu_memory_usage(device):
    if device.type == 'cuda':
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory (GB): Allocated={allocated:.2f}GB, Reserved={reserved:.2f}GB")

In [9]:
def train_wgan_gp(dataloader, generator, critic, g_optimizer, d_optimizer, device, epochs=100, latent_dim=10):
    for epoch in range(epochs):
        print_gpu_memory_usage(device)
        for i, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")):
            real_sequences = batch['sequence'].to(device)

            conditions = batch['condition'].to(device)
            batch_size, seq_len, _ = real_sequences.shape

            for _ in range(CRITIC_ITERATIONS):
                d_optimizer.zero_grad()
                noise = torch.randn(batch_size, latent_dim, device=device)
                fake_sequences = generator(noise, conditions, seq_len)

                real_validity = critic(real_sequences, conditions)
                fake_validity = critic(fake_sequences.detach(), conditions)

                gradient_penalty = compute_gradient_penalty(critic, real_sequences.data, fake_sequences.data, conditions.data, device)

                #print(f"Gradient Penalty: {gradient_penalty.item():.4f}", 'Real_validity:', torch.mean(real_validity).item(), 'Fake_validity:', torch.mean(fake_validity).item())

                d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA_GP * gradient_penalty
                d_loss.backward()
                # critic gradient clipping to avoid exploding gradients
                torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0) 
                d_optimizer.step()

            g_optimizer.zero_grad()
            gen_sequences = generator(noise, conditions, seq_len)
            g_loss = -torch.mean(critic(gen_sequences, conditions))
            g_loss.backward()
            # generator gradient clipping to avoid exploding gradients
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            g_optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}], Critic Loss: {d_loss.item():.4f}, Generator Loss: {g_loss.item():.4f}")

In [10]:
DATA_DIR = "quic_data"
BATCH_SIZE = 128
LATENT_DIM = 20
HIDDEN_DIM = 256
EPOCHS = 2000 # WGANs often need more epochs
LR = 0.0005 # WGANs often use a single LR and the Adam betas below

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [12]:
dataset = QuicSequenceDataset(high_level_csv, low_level_zip_path=low_level_dir) # Provide your file paths
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=4)

condition_dim = dataset.scaled_conditions.shape[1]
sequence_feature_dim = len(dataset.sequence_columns)

generator = Generator(LATENT_DIM, condition_dim, sequence_feature_dim, HIDDEN_DIM, num_layers=2).to(device)
critic = Critic(condition_dim, sequence_feature_dim, HIDDEN_DIM, num_layers=1).to(device)

G_LR = 5e-5 
D_LR = 5e-5

WEIGHT_DECAY = 1e-6

g_optimizer = torch.optim.Adam(
    generator.parameters(), 
    lr=G_LR, 
    betas=(0.0, 0.999),
    weight_decay=WEIGHT_DECAY 
) 
d_optimizer = torch.optim.Adam(
    critic.parameters(), 
    lr=D_LR, 
    betas=(0.0, 0.999),
    weight_decay=WEIGHT_DECAY
)

Loading & Validating Sequences: 100%|██████████| 11897/11897 [00:16<00:00, 708.07it/s]


Dropped 125 sequences containing negative values.
Unscaled Sequence Data Min/Max: 0.0000 / 1398.0000


Scaling Sequences to Tensors: 100%|██████████| 11772/11772 [00:00<00:00, 12642.85it/s]


In [13]:
dataset.condition_features_df.head()

,connection_duration,version_negotiation_occurred,retry_occurred,first_path_validation_response_latency,path_validation_initiated,packets_sent_client,packets_sent_server,handshake_duration,time_to_migration,migration_duration,...,file_id,implementation_aioquic,implementation_nginx,implementation_quicgo,implementation_quiche,migration_type_AFTER_DOWNLOAD,migration_type_BEFORE_DOWNLOAD,migration_type_DURING_DOWNLOAD,migration_type_NO_DOWNLOAD,migration_type_NO_MIGRATION
0,49.246073,0,0,14.654160,1.0,7,7,16.601086,28.691053,15.407085,...,3172,True,False,False,False,False,True,False,False,False
1,22.886992,0,0,9.438992,1.0,8,6,9.938002,9.589911,10.220051,...,3173,True,False,False,False,False,True,False,False,False
2,24.952888,0,0,10.677099,1.0,8,6,10.686874,9.984970,11.665106,...,3174,True,False,False,False,False,True,False,False,False
3,31.651020,0,0,14.860868,1.0,6,5,11.970997,12.630939,15.383959,...,3175,True,False,False,False,False,True,False,False,False
4,35.304070,0,0,14.575958,1.0,7,7,10.221004,17.075062,15.029907,...,3176,True,False,False,False,False,True,False,False,False


In [14]:
dataset.preloaded_sequences

[tensor([[-1.0000,  0.7672, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,  1.0000,
          -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
          -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-0.9998, -0.9537,  1.0000, -1.0000, -1.0000, -1.0000, -1.0000,  1.0000,
          -1.0000,  1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
          -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-0.9990,  0.7672, -1.0000,  1.0000,  1.0000, -1.0000, -1.0000, -1.0000,
          -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
          -1.0000, -1.0000, -0.5000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-0.9980,  0.7672,  1.0000,  1.0000,  1.0000, -1.0000,  1.0000, -1.0000,
          -1.0000, -1.0000,  0.0000,  1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
          -1.0000, -1.0000,  1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1

In [ ]:
train_wgan_gp(dataloader, generator, critic, g_optimizer, d_optimizer, device, epochs=EPOCHS, latent_dim=LATENT_DIM)

GPU Memory (GB): Allocated=0.00GB, Reserved=0.02GB


Epoch 1/2000:   0%|          | 0/92 [00:00<?, ?it/s]/home/ubuntu/.conda/envs/tddpm/lib/python3.9/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch 1/2000: 100%|██████████| 92/92 [00:21<00:00,  4.37it/s]


Epoch [1/2000], Critic Loss: -11.2639, Generator Loss: 6.3056
GPU Memory (GB): Allocated=0.08GB, Reserved=0.67GB


Epoch 2/2000: 100%|██████████| 92/92 [00:20<00:00,  4.54it/s]


Epoch [2/2000], Critic Loss: -6.8129, Generator Loss: 4.7574
GPU Memory (GB): Allocated=0.08GB, Reserved=0.68GB


...

Epoch 872/2000: 100%|██████████| 92/92 [00:19<00:00,  4.69it/s]


Epoch [872/2000], Critic Loss: -36.5674, Generator Loss: 1308.4349
GPU Memory (GB): Allocated=0.08GB, Reserved=0.68GB


Epoch 873/2000: 100%|██████████| 92/92 [00:20<00:00,  4.49it/s]


Epoch [873/2000], Critic Loss: -1.0047, Generator Loss: 1133.9728
GPU Memory (GB): Allocated=0.08GB, Reserved=0.68GB


Epoch 874/2000:  18%|█▊        | 17/92 [00:03<00:15,  4.74it/s]

In [ ]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import joblib

In [ ]:
save_dir = "trained_wgan_quic_no_negative2"
os.makedirs(save_dir, exist_ok=True)

# 2. Define file paths
generator_path = os.path.join(save_dir, "generator.pth")
critic_path = os.path.join(save_dir, "critic.pth")
sequence_scaler_path = os.path.join(save_dir, "sequence_scaler.gz")
condition_scaler_path = os.path.join(save_dir, "condition_scaler.gz")
sequence_columns_path = os.path.join(save_dir, "sequence_columns.json")

# 3. Save the models' state dictionaries
# We save the state_dict, which is just the weights and biases. It's more flexible
# than saving the entire model object.
torch.save(generator.state_dict(), generator_path)
torch.save(critic.state_dict(), critic_path)

# 4. Save the fitted scalers using joblib
joblib.dump(dataset.sequence_scaler, sequence_scaler_path)
joblib.dump(dataset.condition_scaler, condition_scaler_path)

# 5. Save the sequence columns list for later use
import json
with open(sequence_columns_path, 'w') as f:
    json.dump(dataset.sequence_columns, f)

print(f"Models saved to {generator_path} and {critic_path}")
print(f"Scalers saved to {sequence_scaler_path} and {condition_scaler_path}")
print(f"Columns saved to {sequence_columns_path}")

Models saved to trained_wgan_quic_no_negative/generator.pth and trained_wgan_quic_no_negative/critic.pth
Scalers saved to trained_wgan_quic_no_negative/sequence_scaler.gz and trained_wgan_quic_no_negative/condition_scaler.gz
Columns saved to trained_wgan_quic_no_negative/sequence_columns.json
